
# BigQuery Advanced Hands-On Lab — GCP Training Program
### Module 1 (Part 2), Module 2 (Parts 1 & 2) — 5 Hours Total

## Problem Statement

We're an e-commerce company ("ShopIndex") that has outgrown the BigQuery basics from Day 1. Three
new pressures have shown up as the business has grown:

1. **Some of our data lives outside BigQuery entirely** — raw order exports sitting in Cloud
   Storage, and increasingly Parquet/Iceberg files from other teams' pipelines. Do we have to load
   every file before we can query it? *(External Tables & BigLake / Lakehouse for Iceberg)*
2. **Our BigQuery bill is unpredictable month to month**, and we don't know whether we should be on
   pay-per-query or a committed-capacity plan. *(Pricing model, Editions, Reservations & slots)*
3. **We need to prove, for an audit, exactly who can see what** — which teams can query which
   tables, whether a support analyst can see a customer's raw email, and where a number in a
   quarterly report actually came from. *(IAM, RLS, encryption, data lineage)*

On top of that, our data science team wants to **build and evaluate ML models without exporting
data out of BigQuery at all** — churn prediction, revenue forecasting, customer segmentation, and
a semantic product search — using nothing but SQL. *(BigQuery ML, Vector Search)*

This notebook builds up to all of the above against **one running e-commerce dataset**
(`customers`, `products`, `orders`), so every concept builds on the last instead of switching
context between modules.


**Run cells top to bottom, in order.** Each concept has a markdown explanation cell (with a
"🖥️ Check it in the console" pointer) immediately followed by a runnable code cell.

**Before you start:**
- You need a GCP project with billing enabled and the BigQuery, Cloud Storage, BigQuery
  Connection, BigQuery Reservation, Data Lineage, and Dataplex APIs enabled (Step 0 enables these
  for you).
- Have `customers.csv`, `products.csv`, and `orders.json` ready to upload when prompted in Step 2.
- **Cost note:** the core dataset is under 200 total rows across all tables, so on-demand query
  cost across this whole notebook should be a few cents at most. Several sections touch **billed
  services beyond core query/storage pricing** (BQML training, Vertex AI embeddings, slot
  reservations) — every one of these is gated behind an explicit `RUN_*_STEP = True` toggle so
  nothing billed happens by accident.

## Terminology note
As of April 20, 2026, **"BigLake" for Iceberg tables was renamed "Lakehouse for Apache Iceberg"**,
and "BigLake metastore" is now the **"Lakehouse runtime catalog."** APIs, client libraries, CLI
commands, and IAM role names are unchanged — only the console/marketing name changed. This
notebook uses the current name in prose while keeping `WITH CONNECTION` / `biglake` naming intact
in code, since that hasn't changed.



## Configuration

Set your **project ID** and **GCS bucket name** here — every other cell in this notebook
references these variables, so you only configure once.

- `PROJECT_ID` — your GCP project ID (not the project *name*).
- `BUCKET_NAME` — a globally unique GCS bucket name. If it doesn't exist yet, this notebook
  creates it for you in Step 0.
- `DATASET_ID` — the BigQuery dataset name that will hold every lab table.
- `LOCATION` — BigQuery/GCS region. `US` is a safe multi-region default.


In [55]:

PROJECT_ID = "himanshugcpproject"  #@param {type:"string"}
BUCKET_NAME = "mybigqueryadvancebucket007"  #@param {type:"string"}
DATASET_ID = "ecommers"  #@param {type:"string"}
LOCATION = "US"  #@param {type:"string"}

RUN_BQML_STEP = True
RUN_BQML_FORECAST_STEP = True
RUN_VECTOR_SEARCH_STEP = True
RUN_RLS_STEP = True
RUN_ICEBERG_STEP = True   # <-- set True since you're actively working through this section
RUN_RESERVATION_STEP = True
STUDENT_EMAIL = "student1@yourdomain.com"

def TBL(name):
    '''Fully-qualified, backtick-quoted table reference for f-string SQL.'''
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"



print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   {BUCKET_NAME}")
print(f"Dataset:  {DATASET_ID}")
print(f"Location: {LOCATION}")


Project:  himanshugcpproject
Bucket:   mybigqueryadvancebucket007
Dataset:  ecommers
Location: US


## Authenticate and set up clients

In [2]:

from google.colab import auth
auth.authenticate_user()
print("Authenticated.")


Authenticated.


In [3]:

!pip install --quiet google-cloud-bigquery google-cloud-storage google-cloud-bigquery-reservation google-cloud-datacatalog-lineage

from google.cloud import bigquery
from google.cloud import storage
from google.cloud.exceptions import Conflict
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

def run_query(sql, label=None, expect_rows=True):
    '''Run a query/DDL/DML statement and, if it returns rows, show them.'''
    if label:
        print(f"--- {label} ---")
    job = bq_client.query(sql)
    if expect_rows:
        try:
            df = job.result().to_dataframe()
            display(df)
        except Exception:
            job.result()
    else:
        job.result()
    if job.total_bytes_processed is not None:
        print(f"Bytes processed: {job.total_bytes_processed:,}")
    return job

print("BigQuery and Storage clients ready.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.3/111.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.0/89.0 kB 5.0 MB/s eta 0:00:00
BigQuery and Storage clients ready.



## Step 0: Create the BigQuery Dataset, GCS Bucket & Enable APIs

CLI equivalents (for reference — this cell does the same thing via the Python SDK):
```
bq mk --dataset --location={LOCATION} {PROJECT_ID}:{DATASET_ID}
gsutil mb -l {LOCATION} gs://{BUCKET_NAME}/
gcloud services enable bigquery.googleapis.com bigqueryconnection.googleapis.com \
  bigqueryreservation.googleapis.com datalineage.googleapis.com dataplex.googleapis.com \
  --project={PROJECT_ID}
```


In [4]:

!gcloud services enable bigquery.googleapis.com bigqueryconnection.googleapis.com \
  bigqueryreservation.googleapis.com datalineage.googleapis.com dataplex.googleapis.com \
  --project={PROJECT_ID}

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
try:
    bq_client.create_dataset(dataset_ref)
    print(f"Created dataset {DATASET_ID}")
except Conflict:
    print(f"Dataset {DATASET_ID} already exists — continuing.")

bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(BUCKET_NAME, location=LOCATION)
    print(f"Created bucket {BUCKET_NAME}")
else:
    print(f"Bucket {BUCKET_NAME} already exists — continuing.")


Operation "operations/acat.p2-406774515547-cf1e400e-a8dd-4bac-8014-1ca8256a706c" finished successfully.


To take a quick anonymous survey, run:
  $ gcloud survey

Created dataset ecommers
Created bucket mybigqueryadvancebucket007



> Check it in the console: **BigQuery > SQL workspace** → your project → the new
> `training_demo` dataset should now appear in the Explorer panel (empty for now), and
> **Cloud Storage > Buckets** should show your new bucket.



## Step 1: Create the Native Tables

Three tables: `customers`, `products`, and `orders`. The `orders` table has a nested/repeated
column (`ARRAY<STRUCT<...>>`) for `line_items` — this is what lets us demonstrate ARRAY/STRUCT/
UNNEST later. It's also our **partitioned + clustered** table (`PARTITION BY order_date`,
`CLUSTER BY region, status`) for the partitioning module.


**Expected outcome:** three empty tables appear in `training_demo` — no rows yet, since this cell only defines the schema. Rows are loaded in the next section.

In [5]:
run_query(f"""
CREATE OR REPLACE TABLE {TBL("customers")} (
  customer_id      INT64,
  first_name       STRING,
  last_name        STRING,
  email            STRING,
  city             STRING,
  country          STRING,
  region           STRING,
  customer_segment STRING,
  signup_date      DATE,
  churned          BOOL
)
""", label="Create customers table", expect_rows=False)

run_query(f"""
CREATE OR REPLACE TABLE {TBL("products")} (
  product_id   INT64,
  product_name STRING,
  category     STRING,
  subcategory  STRING,
  unit_price   FLOAT64,
  description  STRING
)
""", label="Create products table", expect_rows=False)

run_query(f"""
CREATE OR REPLACE TABLE {TBL("orders")} (
  order_id    INT64,
  customer_id INT64,
  region      STRING,
  order_date  DATE,
  status      STRING,
  line_items  ARRAY<STRUCT<
    product_id   INT64,
    product_name STRING,
    quantity     INT64,
    unit_price   FLOAT64
  >>
)
PARTITION BY order_date
CLUSTER BY region, status
""", label="Create orders table (partitioned + clustered)", expect_rows=False)


--- Create customers table ---
Bytes processed: 0
--- Create products table ---
Bytes processed: 0
--- Create orders table (partitioned + clustered) ---
Bytes processed: 0


QueryJob<project=himanshugcpproject, location=US, id=0f614d99-497a-437c-b795-3f975462873c>


> Check it in the console: **BigQuery > SQL workspace > training_demo** → all three tables
> should be listed. Click **orders** → the **Schema** tab shows `line_items` as a `RECORD` field
> with `REPEATED` mode — that's how the console displays `ARRAY<STRUCT<...>>`.



## Step 2: Upload the Dataset Files and Load Them Into BigQuery

Run the cell below, then select `customers.csv`, `products.csv`, and `orders.json` all at once
from the file picker.


**Expected outcome:** a file-picker dialog opens in the cell's output — select all three files. Nothing else happens until you run the load cell that follows.

In [6]:
from google.colab import files
print("Upload customers.csv, products.csv, and orders.json (select all three).")
uploaded = files.upload()


Upload customers.csv, products.csv, and orders.json (select all three).


Saving customers.csv to customers.csv
Saving products.csv to products.csv
Saving orders.json to orders.json



Now push each file to the GCS bucket, then load it into its matching BigQuery table.
`WRITE_TRUNCATE` makes this safe to re-run — each run replaces the table contents rather than
appending duplicates.


In [7]:
def upload_to_gcs(local_filename):
    blob = bucket.blob(f"{DATASET_ID}/{local_filename}")
    blob.upload_from_filename(local_filename)
    uri = f"gs://{BUCKET_NAME}/{DATASET_ID}/{local_filename}"
    print(f"Uploaded {local_filename} -> {uri}")
    return uri

customers_uri = upload_to_gcs("customers.csv")
products_uri  = upload_to_gcs("products.csv")
orders_uri    = upload_to_gcs("orders.json")


Uploaded customers.csv -> gs://mybigqueryadvancebucket007/ecommers/customers.csv
Uploaded products.csv -> gs://mybigqueryadvancebucket007/ecommers/products.csv
Uploaded orders.json -> gs://mybigqueryadvancebucket007/ecommers/orders.json


In [8]:
csv_job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
# Note: TBL() returns a backtick-quoted string for embedding in SQL (e.g. f"SELECT * FROM {TBL('x')}").
# The Python client methods below want a PLAIN "project.dataset.table" string instead, so we
# strip the backticks with .strip("`") -- forgetting this is a common gotcha with this helper.
bq_client.load_table_from_uri(customers_uri, TBL("customers").strip("`"), job_config=csv_job_config).result()
print("customers loaded:", bq_client.get_table(TBL("customers").strip("`")).num_rows, "rows")

bq_client.load_table_from_uri(products_uri, TBL("products").strip("`"), job_config=csv_job_config).result()
print("products loaded:", bq_client.get_table(TBL("products").strip("`")).num_rows, "rows")

json_job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
bq_client.load_table_from_uri(orders_uri, TBL("orders").strip("`"), job_config=json_job_config).result()
print("orders loaded:", bq_client.get_table(TBL("orders").strip("`")).num_rows, "rows")


customers loaded: 30 rows
products loaded: 26 rows
orders loaded: 70 rows



> Check it in the console: **BigQuery > SQL workspace > training_demo > orders** →
> **Preview** tab → expand any row's `line_items` field to see the nested array rendered as a
> mini-table within the cell — the console's way of showing repeated/nested data without SQL.



## Step 3: Simple SQL Queries

**The question we're answering:** basic exploration — what does a first look at each table tell
us, and can we filter/aggregate meaningfully already?

**3a — Basic SELECT with LIMIT.** Teaching point: always `LIMIT` on a first look at a new table.


**Expected outcome for this section:** three small result tables print inline (via `display()`), each preceded by its `Bytes processed` count — on this small dataset, expect these counts to be in the low thousands of bytes, not megabytes.

In [9]:
run_query(f"""
SELECT customer_id, first_name, last_name, region, customer_segment
FROM {TBL("customers")}
LIMIT 10
""", label="3a. Basic SELECT")


--- 3a. Basic SELECT ---


,customer_id,first_name,last_name,region,customer_segment
0,3,Meera,Iyer,APAC,SMB
1,18,Sanjay,Bhatt,APAC,Enterprise
2,28,Kunal,Rana,APAC,Consumer
3,8,Dev,Reddy,APAC,Consumer
4,13,Divya,Pillai,APAC,Consumer
5,23,Nisha,Chowdhury,APAC,Enterprise
6,17,Kavya,Das,EU,Enterprise
7,22,Varun,D'Souza,EU,SMB
8,7,Sana,Khan,EU,Enterprise
9,12,Arjun,Chatterjee,EU,Enterprise


Bytes processed: 1,126


QueryJob<project=himanshugcpproject, location=US, id=ac54e963-28c6-48f7-9c53-e02ef6a9c015>

**3b — Filtering + sorting.** Reinforces `WHERE` + `ORDER BY` evaluation order.

In [10]:
run_query(f"""
SELECT product_name, category, unit_price
FROM {TBL("products")}
WHERE category = 'Electronics' AND unit_price > 2000
ORDER BY unit_price DESC
""", label="3b. Filter + sort")


--- 3b. Filter + sort ---


,product_name,category,unit_price
0,27-inch Monitor,Electronics,15999.0
1,Smart Watch,Electronics,9999.0
2,Noise Cancelling Headphones,Electronics,8999.0
3,Mechanical Keyboard,Electronics,3499.0
4,Fitness Tracker Band,Electronics,2999.0
5,Bluetooth Speaker,Electronics,2499.0
6,Webcam HD,Electronics,2299.0


Bytes processed: 903


QueryJob<project=himanshugcpproject, location=US, id=ba5690cc-bd47-434a-b40a-4d693865a412>


**3c — Aggregation with `COUNTIF`.** `COUNTIF(condition)` is a BigQuery convenience over
`SUM(CASE WHEN condition THEN 1 ELSE 0 END)`.


In [11]:
run_query(f"""
SELECT region, COUNT(*) AS num_customers, COUNTIF(churned) AS churned_count
FROM {TBL("customers")}
GROUP BY region
ORDER BY num_customers DESC
""", label="3c. Aggregation")


--- 3c. Aggregation ---


,region,num_customers,churned_count
0,APAC,6,2
1,EU,6,3
2,LATAM,6,1
3,MEA,6,0
4,US,6,2


Bytes processed: 186


QueryJob<project=himanshugcpproject, location=US, id=da58b194-e83b-40ea-96fe-f2135804e889>


> Check it in the console: paste any of the three queries above into
> **BigQuery > SQL workspace**'s query editor and click Run — the **Query results** panel lets you
> pin, export, or chart the result directly, and the **Job information** panel shows exact bytes
> processed for each, a preview of the pricing discussion coming up in Step 8.3.



## Step 4: Table Joins (Including UNNEST on Nested `line_items`)

**4a — Standard INNER JOIN** — orders to customers.


**Expected outcome for this section:** 4a shows flat order+customer rows; 4b shows one row per *line item* (more rows than 4a, since one order can have multiple items); 4c shows a revenue rollup with far fewer rows (one per region+category combination).

In [12]:
run_query(f"""
SELECT o.order_id, o.order_date, o.status, c.first_name, c.last_name, c.region
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC
LIMIT 15
""", label="4a. INNER JOIN")


--- 4a. INNER JOIN ---


,order_id,order_date,status,first_name,last_name,region
0,1002,2026-06-25,Completed,Varun,D'Souza,EU
1,1006,2026-06-18,Pending,Dev,Reddy,APAC
2,1017,2026-06-01,Cancelled,Aditya,Kaur,US
3,1001,2026-05-31,Completed,Sanjay,Bhatt,APAC
4,1022,2026-05-23,Completed,Rajesh,Ahluwalia,MEA
5,1047,2026-04-05,Completed,Kavya,Das,EU
6,1064,2026-03-30,Pending,Sanjay,Bhatt,APAC
7,1037,2026-03-26,Returned,Arjun,Chatterjee,EU
8,1055,2026-02-25,Returned,Kunal,Rana,APAC
9,1039,2026-02-06,Completed,Manish,Singh,MEA


Bytes processed: 3,245


QueryJob<project=himanshugcpproject, location=US, id=17f3caf1-8b3e-48c8-b799-509401e91d23>


**4b — Unpack nested `line_items`, then join to `products`** for full order detail. `UNNEST`
behaves like an implicit join against the array, turning nested rows into flat ones.


In [13]:
run_query(f"""
SELECT
  o.order_id,
  o.order_date,
  c.first_name,
  item.product_name,
  item.quantity,
  item.unit_price,
  item.quantity * item.unit_price AS line_total
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id,
UNNEST(o.line_items) AS item
ORDER BY o.order_date DESC
LIMIT 15
""", label="4b. UNNEST + JOIN")


--- 4b. UNNEST + JOIN ---


,order_id,order_date,first_name,product_name,quantity,unit_price,line_total
0,1002,2026-06-25,Varun,Notebook Set,2,349.0,698.0
1,1002,2026-06-25,Varun,Desk Organizer,1,799.0,799.0
2,1002,2026-06-25,Varun,Coffee Maker,2,5999.0,11998.0
3,1006,2026-06-18,Dev,27-inch Monitor,3,15999.0,47997.0
4,1017,2026-06-01,Aditya,Mechanical Keyboard,3,3499.0,10497.0
5,1001,2026-05-31,Sanjay,Electric Kettle,3,1799.0,5397.0
6,1022,2026-05-23,Rajesh,Dumbbell Set,1,3999.0,3999.0
7,1022,2026-05-23,Rajesh,Mechanical Keyboard,1,3499.0,3499.0
8,1047,2026-04-05,Kavya,Backpack,3,2999.0,8997.0
9,1047,2026-04-05,Kavya,Fountain Pen,1,1499.0,1499.0


Bytes processed: 7,337


QueryJob<project=himanshugcpproject, location=US, id=03de47d2-d8be-419f-b326-5364046113d2>

**4c — Full revenue rollup** (orders -> line_items -> products -> customers).

In [14]:
run_query(f"""
SELECT
  c.region,
  p.category,
  SUM(item.quantity * item.unit_price) AS total_revenue
FROM {TBL("orders")} o,
UNNEST(o.line_items) AS item
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id
JOIN {TBL("products")} p ON item.product_id = p.product_id
WHERE o.status = 'Completed'
GROUP BY c.region, p.category
ORDER BY total_revenue DESC
""", label="4c. Full revenue rollup")


--- 4c. Full revenue rollup ---


,region,category,total_revenue
0,LATAM,Furniture,152592.0
1,APAC,Electronics,109490.0
2,APAC,Furniture,90993.0
3,EU,Electronics,63592.0
4,MEA,Fitness,31590.0
5,EU,Kitchen,29393.0
6,MEA,Electronics,28589.0
7,EU,Furniture,23089.0
8,EU,Accessories,18593.0
9,APAC,Kitchen,17792.0


Bytes processed: 3,124


QueryJob<project=himanshugcpproject, location=US, id=43e90852-400f-4661-bcf7-c3d1461b37c6>


> Check it in the console: run 4c in the SQL workspace → click **Execution details**
> (next to Job information) → the execution graph shows each join and the `UNNEST` as separate
> stages, with rows-in/rows-out per stage — useful for spotting which join is actually expensive
> in a bigger, real table.



## Step 5: Views

A view is a saved query — it stores no data; it re-runs the underlying SQL against live data
every time it's queried.

**Advantages:** no storage cost, always fresh, simplifies complex logic for downstream analysts.
**Trade-off:** every query against the view redoes the full join/unnest work — the natural segue
into materialized views (Step 6).


**Expected outcome:** the `CREATE VIEW` cell prints no rows (`expect_rows=False`); the query right after it prints one row per region with a `revenue` figure — confirming the view is live and queryable immediately.

In [15]:
run_query(f"""
CREATE OR REPLACE VIEW {TBL("v_completed_order_details")} AS
SELECT
  o.order_id,
  o.order_date,
  o.region,
  c.customer_segment,
  item.product_name,
  item.quantity,
  item.unit_price,
  item.quantity * item.unit_price AS line_total
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id,
UNNEST(o.line_items) AS item
WHERE o.status = 'Completed'
""", label="Create view", expect_rows=False)

run_query(f"""
SELECT region, SUM(line_total) AS revenue
FROM {TBL("v_completed_order_details")}
GROUP BY region
""", label="Query the view")


--- Create view ---
Bytes processed: 0
--- Query the view ---


,region,revenue
0,MEA,91319.0
1,APAC,223274.0
2,LATAM,161389.0
3,US,37629.0
4,EU,140863.0


Bytes processed: 2,058


QueryJob<project=himanshugcpproject, location=US, id=a161bfc5-3add-4e8d-a244-6569d6e4d906>


> Check it in the console: **training_demo** dataset → `v_completed_order_details` shows a
> small "view" icon distinguishing it from a table → its **Details** tab shows **Table type:
> VIEW** and the exact SQL stored (no data), and its **Schema** tab shows the *computed* columns
> (like `line_total`) as if they were regular columns.



## Step 6: Materialized Views

Unlike a view, a materialized view actually **stores** the precomputed result and incrementally
refreshes it as base tables change — trading storage cost for query speed and lower repeated
compute.

| | View | Materialized View |
|---|---|---|
| Stores data? | No | Yes (precomputed result) |
| Freshness | Always live | Auto-refreshed incrementally |
| Query cost | Full cost each run | Cheap — reads precomputed data |
| Storage cost | None | Yes |
| Best for | Hiding complexity, low-frequency queries | Dashboards, repeated aggregations |

**Caveat:** materialized views have restrictions on supported SQL — check current docs.


**Expected outcome:** similar to Step 5, but note the `num_orders` column now comes from `APPROX_COUNT_DISTINCT` rather than exact `COUNT(DISTINCT ...)` — incremental materialized views don't support exact distinct counts, so a small approximation is the trade-off for the refresh performance this section is demonstrating.

In [16]:
run_query(f"""
CREATE MATERIALIZED VIEW {TBL("mv_region_daily_revenue")} AS
SELECT
  o.region,
  o.order_date,
  SUM(item.quantity * item.unit_price) AS daily_revenue,
  APPROX_COUNT_DISTINCT(o.order_id) AS num_orders  -- incremental MVs don't support COUNT(DISTINCT)
FROM {TBL("orders")} o,
UNNEST(o.line_items) AS item
WHERE o.status = 'Completed'
GROUP BY o.region, o.order_date
""", label="Create materialized view", expect_rows=False)

run_query(f"""
SELECT * FROM {TBL("mv_region_daily_revenue")}
WHERE region = 'APAC'
ORDER BY order_date DESC
""", label="Query the materialized view")


--- Create materialized view ---
Bytes processed: 0
--- Query the materialized view ---


,region,order_date,daily_revenue,num_orders
0,APAC,2026-05-31,5397.0,1
1,APAC,2025-04-17,56996.0,1
2,APAC,2024-11-05,5397.0,1
3,APAC,2024-10-08,79991.0,1
4,APAC,2024-02-06,27496.0,1
5,APAC,2024-01-10,47997.0,1


Bytes processed: 532


QueryJob<project=himanshugcpproject, location=US, id=a60bad6d-662c-41a0-bda3-8cc977cd8454>


> Check it in the console: `mv_region_daily_revenue` shows a distinct "materialized view"
> icon → its **Details** tab has a **Refresh** section showing last refresh time and staleness —
> information a plain view's Details tab never shows.



## Step 7: Partitioning — and Proving the Advantage

The `orders` table is already `PARTITION BY order_date, CLUSTER BY region, status`. Here we
demonstrate *why* that matters, by comparing against an unpartitioned copy.

**7a** — create the unpartitioned copy for comparison.


**Expected outcome:** the 7b cell prints two `Bytes processed` figures back to back — watch for the partitioned version being smaller (partition pruning working), even though at this table's tiny size the difference will look modest in absolute terms.

In [17]:
run_query(f"""
CREATE OR REPLACE TABLE {TBL("orders_unpartitioned")} AS
SELECT * FROM {TBL("orders")}
""", label="Create unpartitioned copy", expect_rows=False)


--- Create unpartitioned copy ---
Bytes processed: 9,296


QueryJob<project=himanshugcpproject, location=US, id=bdfc32cc-73b6-4455-a483-fb4b5a6c51c6>

**7b** — run the same filtered query against both, and compare bytes processed.

In [18]:
print("Partitioned table:")
run_query(f"""
SELECT * FROM {TBL("orders")}
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'
""", label="Filtered query - partitioned")

print("\nUnpartitioned copy:")
run_query(f"""
SELECT * FROM {TBL("orders_unpartitioned")}
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'
""", label="Filtered query - unpartitioned")


Partitioned table:
--- Filtered query - partitioned ---


,order_id,customer_id,region,order_date,status,line_items
0,1040,6,US,2026-01-24,Completed,"[{'product_id': 26, 'product_name': 'Webcam HD..."


Bytes processed: 149

Unpartitioned copy:
--- Filtered query - unpartitioned ---


,order_id,customer_id,region,order_date,status,line_items
0,1040,6,US,2026-01-24,Completed,"[{'product_id': 26, 'product_name': 'Webcam HD..."


Bytes processed: 9,296


QueryJob<project=himanshugcpproject, location=US, id=4f63174c-34f2-4b4b-8a1c-72450eac7365>


Teaching point: the partitioned version processes less data via **partition pruning**. At only
~70 rows this looks small in absolute terms — *"at this toy scale the savings look trivial, but
this exact pruning mechanism is what turns a 500 GB scan into a 2 GB scan on a real production
table."*

**7c** — clustering's advantage stacked on top of partitioning.


In [19]:
run_query(f"""
SELECT * FROM {TBL("orders")}
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'
  AND region = 'APAC'
  AND status = 'Completed'
""", label="Partition + cluster pruning")


--- Partition + cluster pruning ---


,order_id,customer_id,region,order_date,status,line_items


Bytes processed: 0


QueryJob<project=himanshugcpproject, location=US, id=f3d5914f-c3f3-49a7-829e-33025f6e6dca>


> Check it in the console: **training_demo > orders** → **Details** tab → a **Partition**
> section confirms `DATE(order_date)`, and a **Clustering** section lists `region, status` — the
> one place both optimizations are visible on the same table.

**Summary:** cost reduction (billed by bytes scanned), speed, manageability (partition expiration
for auto-deleting old data), and clustering stacking on top for additional pruning dimensions
beyond the one allowed partition column.



## Step 8 (Module 1, Part 2): External Tables, BigLake & Lakehouse for Apache Iceberg

### Problem Statement
Not all of ShopIndex's data should be *loaded* into BigQuery — some of it is owned and maintained
by other teams' pipelines (a Spark job writing Parquet, a partner dropping CSVs into a shared
bucket). Copying it in means paying for duplicate storage and it goes stale the moment the source
updates. **External tables** let BigQuery query data in place; **BigLake** (now marketed as part of
**Lakehouse for Apache Iceberg**) adds governed, uniform access control on top of that external
data — the same fine-grained security you'd get on a native table.

### 8.1 External Table on GCS (JSON)
Query the same `orders.json` directly from GCS — no load step, no bytes copied into BigQuery
managed storage.


**Expected outcome:** the `CREATE OR REPLACE EXTERNAL TABLE` cell prints no rows; the query after it returns the same order data as the native `orders` table, proving BigQuery can read the GCS file directly without a load step.

In [20]:
run_query(f"""
CREATE OR REPLACE EXTERNAL TABLE {TBL("orders_external")}
OPTIONS (
  format = 'NEWLINE_DELIMITED_JSON',
  uris = ['{orders_uri}']
)
""", label="Create external table", expect_rows=False)

run_query(f"""
SELECT order_id, order_date, status
FROM {TBL("orders_external")}
LIMIT 10
""", label="Query the external table")


--- Create external table ---
Bytes processed: 0
--- Query the external table ---


,order_id,order_date,status
0,1001,2026-05-31,Completed
1,1002,2026-06-25,Completed
2,1003,2024-01-04,Completed
3,1004,2025-10-16,Cancelled
4,1005,2024-01-01,Completed
5,1006,2026-06-18,Pending
6,1007,2024-03-11,Completed
7,1008,2025-06-24,Pending
8,1009,2025-10-27,Completed
9,1010,2025-08-25,Returned


Bytes processed: 3,697


QueryJob<project=himanshugcpproject, location=US, id=744a8981-ef3a-4d0f-ad70-146d14608f01>


> Check it in the console: **training_demo > orders_external** → the **Details** tab shows
> **Table type: EXTERNAL** and a **Source URI(s)** field pointing at your GCS path — confirming no
> data was copied, only a reference was created. Compare its **Storage** stats (should show ~0
> bytes) against `orders`'s.

`orders_external` is queried exactly like a native table, but the underlying bytes stay in GCS.
This is the foundation for BigLake — **without** a `WITH CONNECTION` clause, this is a plain
external table (query-time-only IAM enforcement, no fine-grained access control). Adding
`WITH CONNECTION` turns it into a BigLake table, unlocking the same row/column-level security
we'll use on native tables later in this notebook.

### 8.2 Managed Apache Iceberg Table (Lakehouse for Apache Iceberg)
**Why this is different from 8.1:** an external table just *points at* existing files. A managed
Iceberg table is a full **BigQuery-managed table** that happens to store its data as open-format
Parquet files (with Iceberg metadata) in *your own* Cloud Storage bucket — giving you BigQuery's
DML, time travel, and access control, while the raw files remain portable to any Iceberg-compatible
engine (Spark, Trino, etc.).

**Setup required first (once per project):** a Cloud resource connection, so BigQuery has
permission to write to your bucket on your behalf.
```
bq mk --connection --connection_type=CLOUD_RESOURCE --location={LOCATION} iceberg_conn
```
Then grant the connection's service account (shown in the command's output) the **Storage Object
Admin** role on your bucket, via **IAM & Admin > IAM** in the console. Once that's done, set
`RUN_ICEBERG_STEP = True` in the configuration cell above.


In [34]:
import time
import json
from google.cloud import bigquery



def TBL(name):
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"

bq_client = bigquery.Client(project=PROJECT_ID)

def run_query(sql, label=None, expect_rows=True):
    if label:
        print(f"--- {label} ---")
    job = bq_client.query(sql)
    if expect_rows:
        try:
            display(job.result().to_dataframe())
        except Exception:
            job.result()
    else:
        job.result()
    return job

if RUN_ICEBERG_STEP:
    CONNECTION_NAME = "iceberg_conn"
    ICEBERG_CONNECTION = f"{PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}"

    # 1. Create the connection if it doesn't already exist (idempotent)
    existing = !bq show --connection {ICEBERG_CONNECTION} 2>&1
    existing_text = "\n".join(existing).lower()
    if "not found" in existing_text or "notfound" in existing_text:
        !bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} \
            --location={LOCATION} {CONNECTION_NAME}
        print(f"Created connection {ICEBERG_CONNECTION}")
    else:
        print(f"Connection {ICEBERG_CONNECTION} already exists — reusing it.")

    # 2. Extract the connection's service account directly from bq show (no guessing)
    raw = !bq show --format=prettyjson --connection {ICEBERG_CONNECTION}
    connection_info = json.loads("\n".join(raw))
    connection_sa_email = connection_info["cloudResource"]["serviceAccountId"]
    print("Connection service account:", connection_sa_email)

    # 3. Grant it permission on the correct bucket (idempotent -- safe to re-run)
    !gsutil iam ch serviceAccount:{connection_sa_email}:roles/storage.objectAdmin gs://{BUCKET_NAME}
    print(f"Granted roles/storage.objectAdmin to {connection_sa_email} on gs://{BUCKET_NAME}")

    # 4. Wait for IAM propagation
    print("Waiting 45s for IAM propagation...")
    time.sleep(45)

    # 5. Now create and populate the Iceberg table
    run_query(f"""
    CREATE OR REPLACE TABLE {TBL("orders_iceberg")} (
      order_id    INT64,
      customer_id INT64,
      region      STRING,
      order_date  DATE,
      status      STRING
    )
    WITH CONNECTION `{ICEBERG_CONNECTION}`
    OPTIONS (
      file_format = 'PARQUET',
      table_format = 'ICEBERG',
      storage_uri = 'gs://{BUCKET_NAME}/{DATASET_ID}/iceberg_warehouse/orders_iceberg'
    )
    """, label="Create managed Iceberg table", expect_rows=False)

    run_query(f"""
    INSERT INTO {TBL("orders_iceberg")}
    SELECT order_id, customer_id, region, order_date, status
    FROM {TBL("orders")}
    """, label="Populate Iceberg table from orders", expect_rows=False)

    run_query(f"""
    SELECT region, COUNT(*) AS num_orders
    FROM {TBL("orders_iceberg")}
    GROUP BY region
    ORDER BY num_orders DESC
    """, label="Query the Iceberg table")
else:
    print("RUN_ICEBERG_STEP is False.")

Connection himanshugcpproject.US.iceberg_conn already exists — reusing it.
Connection service account: bqcx-406774515547-xhi5@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Granted roles/storage.objectAdmin to bqcx-406774515547-xhi5@gcp-sa-bigquery-condel.iam.gserviceaccount.com on gs://mybigqueryadvancebucket007
Waiting 45s for IAM propagation...
--- Create managed Iceberg table ---
--- Populate Iceberg table from orders ---
--- Query the Iceberg table ---


,region,num_orders
0,APAC,18
1,MEA,17
2,US,13
3,EU,12
4,LATAM,10



> Check it in the console: **training_demo > orders_iceberg** → **Details** tab shows
> **Table type: EXTERNAL** but with a **BigLake** badge and the connection name, plus
> **Storage format: ICEBERG** — and in **Cloud Storage**, your bucket's `iceberg_warehouse/`
> prefix now contains actual Parquet data files and Iceberg metadata JSON, which you (or a Spark
> job) could read directly, outside BigQuery entirely.

**When to reach for this vs. a native table:** if another engine (Spark, a partner's Trino
cluster) needs to read the *same* data without going through BigQuery at all, Iceberg managed
tables give you that portability. If BigQuery is the only consumer, a plain native table is
simpler and has no extra connection/IAM setup.



## Step 8.3 (Module 1, Part 2): BigQuery Pricing — On-Demand vs. Editions & Reservations

### Problem Statement
ShopIndex has been on **on-demand pricing** since day one — pay per byte scanned, no upfront
commitment. As query volume has grown, Finance wants predictable monthly costs instead of a bill
that swings with usage, and the data team wants some workloads (nightly ETL) to never compete for
capacity with an executive's ad-hoc dashboard query. This is exactly the on-demand-vs-reservations
decision.

### 8.3.1 The Two Pricing Models
| | On-Demand | Editions + Reservations |
|---|---|---|
| **You pay for** | Bytes scanned per query | A committed pool of **slots** (units of query compute) |
| **Cost predictability** | Variable — scales with usage | Fixed baseline, optional autoscaling burst |
| **Best for** | Spiky, low-to-moderate volume workloads (most training/demo usage, including this notebook) | Steady, high-volume production workloads |
| **Commitment** | None | Annual or 3-year capacity commitments (for the best per-slot rate); reservations themselves can autoscale slot-by-slot on top of a baseline |
| **Editions** | N/A | **Standard**, **Enterprise**, **Enterprise Plus** — each unlocking different feature sets (e.g., RLS and column-level security require at least Enterprise) |

**This is why every query in this notebook has been billed on-demand** — appropriate for a small
training dataset, and exactly the plain default every new BigQuery project starts on.

### 8.3.2 Inspect Reservations & Commitments (Read-Only — Safe to Run Regardless of the Toggle)
**What this does:** lists whatever reservations/commitments already exist in your project — this
is purely informational and costs nothing, unlike actually creating a commitment.


**Expected outcome:** on a fresh training project, both lists will almost always print empty — that's the expected, correct result, confirming you're on plain on-demand pricing (the default for every new project).

In [ ]:
from google.cloud import bigquery_reservation_v1

reservation_client = bigquery_reservation_v1.ReservationServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

print("Existing capacity commitments:")
for commitment in reservation_client.list_capacity_commitments(parent=parent):
    print(f"  {commitment.name} — {commitment.slot_count} slots, {commitment.state.name}")

print("\nExisting reservations:")
for reservation in reservation_client.list_reservations(parent=parent):
    print(f"  {reservation.name} — {reservation.slot_capacity} slots")

print("\nIf both lists are empty, your project is on pure on-demand pricing (the default) —")
print("exactly what this notebook has been using throughout.")



### 8.3.3 Creating a Reservation — Reference Syntax Only (Not Executed by Default)
**Why this is reference-only for a training lab:** capacity commitments require **Enterprise or
Enterprise Plus edition** and a minimum purchase (annual or 3-year plans), which is real, ongoing
money you don't want a class accidentally triggering. Shown here so students see the actual
commands, gated behind `RUN_RESERVATION_STEP`.


In [ ]:
if RUN_RESERVATION_STEP:
    print("Reference commands only — uncomment deliberately if you truly intend to purchase capacity:")
    print(f"""
# 1. Purchase a capacity commitment (real money, annual/3-year lock-in):
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --capacity_commitment=true \\
#   --edition=ENTERPRISE --plan=ANNUAL --renewal_plan=NONE --slots=100

# 2. Create a reservation drawing from that commitment:
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --reservation=true \\
#   --slots=100 --edition=ENTERPRISE training-demo-reservation

# 3. Assign this project to the reservation (without this, it stays on-demand):
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --reservation_assignment=true \\
#   --reservation_id=training-demo-reservation --job_type=QUERY \\
#   --assignee_type=PROJECT --assignee_id={PROJECT_ID}
""")
else:
    print("RUN_RESERVATION_STEP is False — this section is reference syntax only by design.")



> Check it in the console: **BigQuery > Administration > Workload management** (or search
> "Workload management") → the **Slot reservations** and **Commitments** tabs show exactly what
> the `list_capacity_commitments`/`list_reservations` calls above returned — this is the console
> equivalent, and where you'd actually click "Buy slots" in a real purchasing decision rather than
> running the CLI commands live.



## Step 8.4 (Module 1, Part 2): IAM & Data Access Control

### Problem Statement
Before getting into *row-level* security (Step 14) or *column-level* encryption (Step 15), there's
a coarser, foundational layer: **who can see this dataset or table at all.** IAM in BigQuery works
at three levels — **project**, **dataset**, and **table/view** — and getting this right is the
prerequisite everything else builds on.

### 8.4.1 The Three Levels, and BigQuery's Predefined Roles
| Level | Grants access to | Common predefined roles |
|---|---|---|
| **Project** | Every dataset in the project (unless overridden) | `roles/bigquery.admin`, `roles/bigquery.dataViewer`, `roles/bigquery.jobUser` |
| **Dataset** | Every table/view in that one dataset | `roles/bigquery.dataViewer`, `roles/bigquery.dataEditor`, `roles/bigquery.dataOwner` |
| **Table/View** | Just that one resource | Same roles, scoped narrower |

**Key distinction to teach:** `roles/bigquery.dataViewer` lets someone *read* table data, but
`roles/bigquery.jobUser` is what actually lets them *run a query job* at all — a common gotcha is
granting Data Viewer without Job User and wondering why the user still can't query anything.

### 8.4.2 Inspect Current IAM Policy on Our Dataset
**What this does:** reads back exactly who has what role on `training_demo` right now — the
starting point before granting anything new.


**Expected outcome:** at least one access entry prints (typically your own account as `OWNER`, plus any project-level roles inherited by the dataset) — an empty result would be unusual and worth double-checking your `PROJECT_ID`/`DATASET_ID` values.

In [35]:
dataset = bq_client.get_dataset(f"{PROJECT_ID}.{DATASET_ID}")
print(f"Access entries on dataset {DATASET_ID}:")
for entry in dataset.access_entries:
    role = entry.role or "(implied by entity type)"
    print(f"  {entry.entity_type}: {entry.entity_id}  ->  role: {role}")


Access entries on dataset ecommers:
  specialGroup: projectWriters  ->  role: WRITER
  specialGroup: projectOwners  ->  role: OWNER
  userByEmail: himanshu.rathi@talktotechie.com  ->  role: OWNER
  specialGroup: projectReaders  ->  role: READER



### 8.4.3 Grant Read-Only Access to a Specific Table
**What this does:** grants a hypothetical analyst read access to just `v_completed_order_details`
(the view from Step 5) — not the entire dataset, and not the raw `customers`/`orders` tables
underneath it. This is the IAM-level analogue of the authorized-view pattern: exposing a curated
slice without exposing everything behind it.


In [36]:
table_ref = bq_client.get_table(TBL("v_completed_order_details").strip("`"))
policy = bq_client.get_iam_policy(table_ref)
policy.bindings.append({
    "role": "roles/bigquery.dataViewer",
    "members": {f"user:{STUDENT_EMAIL}"},
})
# Uncomment to actually apply -- left inert by default since STUDENT_EMAIL is a placeholder
# unless you've set it to a real principal in your domain:
# bq_client.set_iam_policy(table_ref, policy)

print(f"Would grant roles/bigquery.dataViewer on v_completed_order_details to {STUDENT_EMAIL}.")
print("Uncomment the set_iam_policy call above once STUDENT_EMAIL is a real user/group in your domain.")


Would grant roles/bigquery.dataViewer on v_completed_order_details to student1@yourdomain.com.
Uncomment the set_iam_policy call above once STUDENT_EMAIL is a real user/group in your domain.



> Check it in the console: **training_demo > v_completed_order_details** → click **Sharing
> > Permissions** (top right of the table page) → this shows the exact same IAM bindings the code
> above reads/writes, in the console's role-picker UI — the most common way this is actually
> managed day to day, rather than via the Python client.

### 8.4.4 IAM vs. Row-Level Security vs. Column-Level Security — When to Use Which
| Mechanism | Controls | Granularity | Covered in this notebook |
|---|---|---|---|
| **IAM** | Access to a whole dataset/table/view | Coarse — all-or-nothing per resource | This section |
| **Row-Level Security (RLS)** | *Which rows* a query returns, even with table access granted | Row | Step 14 |
| **Column-level security / masking** | Whether a specific column's *real value* is visible | Column | Step 15 |

These three **stack** — a user needs IAM access to query the table at all; RLS then filters which
rows they see; column masking then determines whether specific columns within those rows show
real or masked values.



## Step 9 (Module 2, Part 1): BigQuery ML — Four Use Cases in SQL

### Problem Statement
ShopIndex's data science team wants four different, common business questions answered — and
wants to answer all of them **without ever exporting data out of BigQuery** into a separate ML
environment. BQML trains and serves models using the exact same SQL dialect used everywhere else
in this notebook.

**Cost note:** every BQML training step below is a billed operation beyond simple query/storage
pricing. All four are gated behind `RUN_BQML_STEP` (and `RUN_BQML_FORECAST_STEP` for the time
series one specifically, since it has its own separate cost/behavior caveats) — set these to
`True` in the configuration cell above before running.

### 9.1 Use Case 1 — Logistic Regression: Predict Customer Churn
**The question:** "Which customers are likely to churn, based on their segment, region, and
purchase history?" — a binary classification problem, the most common BQML starting point.


**Expected outcome (only if `RUN_BQML_STEP = True`):** training takes roughly 1-2 minutes on this small dataset; `ML.EVALUATE` then prints one row with columns like `precision`, `recall`, `roc_auc` — don't expect impressive numbers on synthetic data this small, the point is the mechanism, not real predictive power.

In [37]:
if RUN_BQML_STEP:
    run_query(f"""
    CREATE OR REPLACE MODEL {TBL("churn_model")}
    OPTIONS(model_type='logistic_reg', input_label_cols=['churned']) AS
    SELECT
      c.customer_segment,
      c.region,
      COUNT(o.order_id) AS num_orders,
      IFNULL(SUM(item.quantity * item.unit_price), 0) AS lifetime_value,
      c.churned
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_segment, c.region, c.churned, c.customer_id
    """, label="Train churn_model", expect_rows=False)

    run_query(f"""
    SELECT * FROM ML.EVALUATE(MODEL {TBL("churn_model")})
    """, label="Evaluate churn_model")
else:
    print("RUN_BQML_STEP is False — set it to True above to train the model live.")


--- Train churn_model ---
--- Evaluate churn_model ---


,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.714286,0.625,0.833333,0.666667,0.267229,0.949539



> Check it in the console: **training_demo > churn_model** → the **Evaluation** tab shows
> precision/recall/ROC curve visually, and the **Training** tab shows the loss curve across
> iterations — both computed automatically, no separate plotting code needed.

### 9.2 Use Case 2 — Linear Regression: Predict Customer Lifetime Value
**The question:** "Given a customer's segment, region, and tenure, what lifetime value should we
expect?" — useful for flagging under-performing segments or setting acquisition-cost targets.
Unlike churn (a yes/no label), lifetime value is a continuous number — `linear_reg`, not
`logistic_reg`.


In [38]:
if RUN_BQML_STEP:
    run_query(f"""
    CREATE OR REPLACE MODEL {TBL("ltv_model")}
    OPTIONS(model_type='linear_reg', input_label_cols=['lifetime_value']) AS
    SELECT
      c.customer_segment,
      c.region,
      DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days,
      IFNULL(SUM(item.quantity * item.unit_price), 0) AS lifetime_value
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_segment, c.region, c.signup_date, c.customer_id
    """, label="Train ltv_model", expect_rows=False)

    run_query(f"""
    SELECT * FROM ML.EVALUATE(MODEL {TBL("ltv_model")})
    """, label="Evaluate ltv_model (look at r2_score and mean_absolute_error)")

    run_query(f"""
    SELECT
      customer_segment, region, tenure_days,
      predicted_lifetime_value
    FROM ML.PREDICT(MODEL {TBL("ltv_model")}, (
      SELECT
        c.customer_segment, c.region,
        DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
      FROM {TBL("customers")} c
      LIMIT 10
    ))
    """, label="Predict LTV for 10 customers")
else:
    print("RUN_BQML_STEP is False — set it to True above to train the model live.")


--- Train ltv_model ---
--- Evaluate ltv_model (look at r2_score and mean_absolute_error) ---


,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,39380.285264,2.490346e+09,15.829545,35741.504902,0.186753,0.187198


--- Predict LTV for 10 customers ---


,customer_segment,region,tenure_days,predicted_lifetime_value
0,SMB,APAC,684,49730.494088
1,Enterprise,APAC,665,68203.766764
2,Consumer,APAC,1176,100172.747675
3,Consumer,APAC,1129,98696.251308
4,Consumer,APAC,697,85125.050649
5,Enterprise,APAC,1071,80958.182197
6,Enterprise,EU,1074,37348.495098
7,SMB,EU,878,12121.036910
8,Enterprise,EU,1125,38950.650731
9,Enterprise,EU,1208,41558.080487



**Teaching point:** `ML.EVALUATE` returns *different* metric columns depending on model type —
`precision`/`recall`/`roc_auc` for classification (9.1), `r2_score`/`mean_absolute_error` for
regression (9.2). Worth pointing out explicitly, since students often expect the same output shape
regardless of model type.

### 9.3 Use Case 3 — K-Means Clustering: Customer Segmentation
**The question:** "Do our customers naturally fall into groups we haven't explicitly labeled?" —
unsupervised learning: no `input_label_cols`, since there's no "correct answer" column to predict.
`KMEANS` groups customers by similarity across the given features.


In [40]:
if RUN_BQML_STEP:
    run_query(f"""
    CREATE OR REPLACE MODEL {TBL("customer_segments_model")}
    OPTIONS(model_type='kmeans', num_clusters=3) AS
    SELECT
      COUNT(o.order_id) AS num_orders,
      IFNULL(SUM(item.quantity * item.unit_price), 0) AS total_spend,
      DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_id, c.signup_date
    """, label="Train customer_segments_model", expect_rows=False)

    run_query(f"""
    SELECT centroid_id, feature, numerical_value
    FROM ML.CENTROIDS(MODEL {TBL("customer_segments_model")})
    ORDER BY centroid_id, feature
    """, label="Inspect the 3 cluster centroids")

    run_query(f"""
    SELECT
      c.customer_id, c.customer_segment,
      CENTROID_ID
    FROM ML.PREDICT(MODEL {TBL("customer_segments_model")}, (
      SELECT
        c.customer_id,
        COUNT(o.order_id) AS num_orders,
        IFNULL(SUM(item.quantity * item.unit_price), 0) AS total_spend,
        DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
      FROM {TBL("customers")} c
      LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
      LEFT JOIN UNNEST(o.line_items) AS item
      GROUP BY c.customer_id, c.signup_date
    )) AS predicted
    JOIN {TBL("customers")} c ON predicted.customer_id = c.customer_id
    ORDER BY CENTROID_ID
    """, label="Which cluster does each customer fall into?")
else:
    print("RUN_BQML_STEP is False — set it to True above to train the model live.")

--- Train customer_segments_model ---
--- Inspect the 3 cluster centroids ---


,centroid_id,feature,numerical_value
0,1,num_orders,16.000000
1,1,tenure_days,1089.500000
2,1,total_spend,195866.000000
3,2,num_orders,4.733333
4,2,tenure_days,724.200000
5,2,total_spend,44981.000000
6,3,num_orders,4.769231
7,3,tenure_days,1121.538462
8,3,total_spend,38282.923077


--- Which cluster does each customer fall into? ---


,customer_id,customer_segment,CENTROID_ID
0,15,Consumer,1
1,8,Consumer,1
2,29,Enterprise,2
3,20,Enterprise,2
4,13,Consumer,2
5,22,SMB,2
6,25,Enterprise,2
7,5,Consumer,2
8,24,Consumer,2
9,14,Consumer,2



**What to look for:** compare `ML.CENTROIDS`' output against the predicted cluster assignments —
one cluster should show high `total_spend`/`num_orders` (your best customers), another low on
both (at-risk or new), giving marketing a data-driven segmentation instead of guessing at
thresholds.

> Check it in the console: **training_demo > customer_segments_model** → the **Evaluation**
> tab shows a cluster visualization (for 2-3 numeric features) — a quick visual sanity check that
> the clusters are actually separated, not overlapping noise.

### 9.4 Use Case 4 — ARIMA_PLUS: Forecast Daily Revenue
**The question:** "Based on order history, what should we expect revenue to look like over the
next 2 weeks?" — time series forecasting.

**Honest caveat for the class:** our `orders` table only spans a few months of synthetic data —
nowhere near the "at least two full seasonal cycles" ARIMA_PLUS wants for a *reliable* forecast.
This section demonstrates the **mechanism** (train -> forecast -> interpret), not a forecast you
should actually trust at this data volume — the same "mechanism over magnitude" caveat from the
partitioning section (Step 7) applies here too.


In [41]:
if RUN_BQML_FORECAST_STEP:
    run_query(f"""
    CREATE OR REPLACE MODEL {TBL("revenue_forecast_model")}
    OPTIONS(
      model_type='ARIMA_PLUS',
      time_series_timestamp_col='order_date',
      time_series_data_col='daily_revenue',
      auto_arima=TRUE
    ) AS
    SELECT
      o.order_date,
      SUM(item.quantity * item.unit_price) AS daily_revenue
    FROM {TBL("orders")} o,
    UNNEST(o.line_items) AS item
    WHERE o.status = 'Completed'
    GROUP BY o.order_date
    """, label="Train revenue_forecast_model", expect_rows=False)

    run_query(f"""
    SELECT * FROM ML.ARIMA_EVALUATE(MODEL {TBL("revenue_forecast_model")})
    """, label="Evaluate candidate ARIMA models")

    run_query(f"""
    SELECT
      forecast_timestamp, forecast_value,
      prediction_interval_lower_bound, prediction_interval_upper_bound
    FROM ML.FORECAST(MODEL {TBL("revenue_forecast_model")}, STRUCT(14 AS horizon, 0.95 AS confidence_level))
    ORDER BY forecast_timestamp
    """, label="14-day revenue forecast")
else:
    print("RUN_BQML_FORECAST_STEP is False — set it to True above to train the model live.")
    print("Remember the small-dataset caveat above before treating the forecast as reliable.")


--- Train revenue_forecast_model ---
--- Evaluate candidate ARIMA models ---


,non_seasonal_p,non_seasonal_d,non_seasonal_q,has_drift,log_likelihood,AIC,variance,seasonal_periods,has_holiday_effect,has_spikes_and_dips,has_step_changes,error_message
0,1,0,0,False,-421.582545,849.165091,2.489079e+08,[NO_SEASONALITY],False,True,False,
1,1,0,1,False,-421.392604,850.785208,2.462013e+08,[NO_SEASONALITY],False,True,False,
2,2,0,0,False,-421.483563,850.967126,2.475258e+08,[NO_SEASONALITY],False,True,False,
3,0,0,2,False,-422.891322,853.782643,2.668602e+08,[NO_SEASONALITY],False,True,False,
4,0,0,1,False,-424.226842,854.453684,2.863661e+08,[NO_SEASONALITY],False,True,False,
5,0,0,0,False,-435.065106,874.130212,5.153137e+08,[NO_SEASONALITY],False,True,False,
6,0,0,0,True,0.000000,0.000000,0.000000e+00,[],False,False,False,Model training failed on the input time series...
7,0,0,1,True,0.000000,0.000000,0.000000e+00,[],False,False,False,Model training failed on the input time series...
8,0,0,2,True,0.000000,0.000000,0.000000e+00,[],False,False,False,Model training failed on the input time series...
9,1,0,0,True,0.000000,0.000000,0.000000e+00,[],False,False,False,Model training failed on the input time series...


--- 14-day revenue forecast ---


,forecast_timestamp,forecast_value,prediction_interval_lower_bound,prediction_interval_upper_bound
0,2026-06-28 00:00:00+00:00,17272.111879,-13594.565830,48138.789588
1,2026-07-22 00:00:00+00:00,19934.968462,-17831.295047,57701.231971
2,2026-08-15 00:00:00+00:00,21812.277213,-18951.064477,62575.618903
3,2026-09-08 00:00:00+00:00,23135.776259,-19038.017431,65309.569949
4,2026-10-02 00:00:00+00:00,24068.840531,-18788.713854,66926.394917
5,2026-10-26 00:00:00+00:00,24726.649042,-18466.722982,67920.021066
6,2026-11-19 00:00:00+00:00,25190.402773,-18168.910503,68549.716048
7,2026-12-13 00:00:00+00:00,25517.348257,-17924.205700,68958.902215
8,2027-01-06 00:00:00+00:00,25747.844193,-17734.527348,69230.215734
9,2027-01-30 00:00:00+00:00,25910.343382,-17592.301159,69412.987924



> Check it in the console: **training_demo > revenue_forecast_model** → the **Evaluation**
> tab plots the forecast with confidence interval bands directly — a genuinely useful visual for
> explaining forecast uncertainty to a non-technical stakeholder, without writing any charting
> code yourself.

**Summary across all four BQML use cases:** the pattern is always `CREATE MODEL ... OPTIONS(...) AS
SELECT ...` to train, `ML.EVALUATE` to check quality, and either `ML.PREDICT` (classification/
regression/clustering) or `ML.FORECAST` (time series) to get results — the same three-step shape
regardless of which of the ~20 model types BQML supports.



## Step 10 (Module 2, Part 1): Time Travel / Change History

### Problem Statement
Someone on the team ran a bad `UPDATE` (or worse, dropped a table) 20 minutes ago. Can we see —
or recover — what existed before that happened, without restoring from a separate backup system?

### 10.1 Time Travel Across an UPDATE


**Expected outcome:** the first query shows the order's original status; after the `UPDATE`, the second query shows `Cancelled`; the time-travel query in the next cell then shows the *original* status again, even though the table has already changed — proof the historical read genuinely bypasses the current state.

In [42]:
before_df = run_query(f"""
SELECT order_id, status FROM {TBL("orders")} WHERE order_id = 1001
""", label="Status before update")

run_query(f"""
UPDATE {TBL("orders")} SET status = 'Cancelled' WHERE order_id = 1001
""", label="Update order 1001", expect_rows=False)

run_query(f"""
SELECT order_id, status FROM {TBL("orders")} WHERE order_id = 1001
""", label="Status after update")


--- Status before update ---


,order_id,status
0,1001,Completed


--- Update order 1001 ---
--- Status after update ---


,order_id,status
0,1001,Cancelled


QueryJob<project=himanshugcpproject, location=US, id=da2cbdbc-4cca-4c3c-b1d6-f5050a0cc141>

In [43]:
# Time travel — query the table as it looked a few minutes ago, before the UPDATE above.
run_query(f"""
SELECT order_id, status
FROM {TBL("orders")}
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 5 MINUTE)
WHERE order_id = 1001
""", label="Time travel query")


--- Time travel query ---


,order_id,status
0,1001,Completed


QueryJob<project=himanshugcpproject, location=US, id=066cbd04-1025-4e49-9e89-14cb63a094ca>


> Check it in the console: **training_demo > orders** → click the **more actions** menu
> → this doesn't expose time travel directly, but you can run the exact same `FOR SYSTEM_TIME
> AS OF` query in the SQL workspace and see the pre-update value returned — worth demonstrating
> live since it looks almost like magic the first time.

### 10.2 Recovering a Dropped Table With `UNDROP`
**What this demonstrates:** time travel isn't just for row-level changes — it also covers
recovering an entire table that was accidentally deleted, within BigQuery's time travel window
(default 7 days, configurable per dataset).


In [53]:
import time
from google.cloud.bigquery import TableReference

run_query(f"""
CREATE OR REPLACE TABLE {TBL("scratch_table_for_undrop_demo")} AS
SELECT * FROM {TBL("products")} LIMIT 5
""", label="Re-create the scratch table", expect_rows=False)

time.sleep(10)
snapshot_time_ms = int(time.time() * 1000)

run_query(f"""
DROP TABLE {TBL("scratch_table_for_undrop_demo")}
""", label="Drop it (simulating an accident)", expect_rows=False)

run_query(f"""
SELECT table_name FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
WHERE table_name = 'scratch_table_for_undrop_demo'
""", label="Confirm it's really gone")

# Recover via a COPY JOB from the snapshot decorator -- not a SQL SELECT. BigQuery's query
# engine doesn't support reading a @timestamp decorator off an already-deleted table; only
# the table-copy operation does. This matches Google's own documented recovery pattern.
source_table_ref = TableReference.from_string(
    f"{PROJECT_ID}.{DATASET_ID}.scratch_table_for_undrop_demo@{snapshot_time_ms}"
)
dest_table_ref = TableReference.from_string(
    f"{PROJECT_ID}.{DATASET_ID}.scratch_table_for_undrop_demo"
)

copy_job = bq_client.copy_table(source_table_ref, dest_table_ref)
copy_job.result()
print("Recovered via copy job from snapshot decorator.")

run_query(f"""
SELECT * FROM {TBL("scratch_table_for_undrop_demo")}
""", label="Confirm the data is back")

--- Re-create the scratch table ---
--- Drop it (simulating an accident) ---
--- Confirm it's really gone ---


,table_name


Recovered via copy job from snapshot decorator.
--- Confirm the data is back ---


,product_id,product_name,category,subcategory,unit_price,description
0,15,Water Bottle,Accessories,Lifestyle,599.0,An insulated stainless steel water bottle that...
1,14,Backpack,Accessories,Bags,2999.0,A water resistant backpack with a padded lapto...
2,2,Mechanical Keyboard,Electronics,Accessories,3499.0,A durable mechanical keyboard with tactile swi...
3,3,27-inch Monitor,Electronics,Displays,15999.0,A 27-inch QHD monitor with vibrant colors and ...
4,1,Wireless Mouse,Electronics,Accessories,799.0,A compact wireless mouse with ergonomic design...


QueryJob<project=himanshugcpproject, location=US, id=16019d3e-c506-40ec-ae22-d62b4e828c13>


**Teaching point:** `UNDROP` only works within the time travel window and only if nothing else
has since been created with the exact same name — worth mentioning as the practical limit of this
safety net, not an infinite undo button.



## Step 11 (Module 2, Part 1): Vector Search for RAG

### Problem Statement
A customer support chatbot needs to find semantically similar products given a natural-language
description — not keyword matching, but "which products *mean* something similar to this query."

This step needs a **remote connection to a Vertex AI embedding model**, created once per project:
```
!bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} \
    --location={LOCATION} embedding_conn
```
Then grant the connection's service account the Vertex AI User role in IAM. Once done, set
`RUN_VECTOR_SEARCH_STEP = True` above.

**Honest caveat:** BigQuery's vector index at very small data scales (like our 26 products) may
fall back to brute-force search rather than using the index — index behavior is tuned for much
larger tables.


In [56]:
!bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} \
    --location={LOCATION} embedding_conn

Connection 406774515547.us.embedding_conn successfully created


In [58]:
!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

Operation "operations/acat.p2-406774515547-58e928c8-fa32-4642-bab3-aed1e9f5d0b2" finished successfully.


In [63]:
import json
import time
from google.api_core.exceptions import BadRequest, NotFound

# ---- 1. Enable required APIs (idempotent -- no-op if already enabled) ----
!gcloud services enable aiplatform.googleapis.com bigqueryconnection.googleapis.com \
    --project={PROJECT_ID} --quiet

# ---- 2. Create the connection if it doesn't already exist ----
CONNECTION_NAME = "embedding_conn"
existing = !bq show --connection {PROJECT_ID}.{LOCATION}.{CONNECTION_NAME} 2>&1
existing_text = "\n".join(existing).lower()
if "not found" in existing_text or "notfound" in existing_text:
    !bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} \
        --location={LOCATION} {CONNECTION_NAME}
    print(f"Created connection {PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}")
else:
    print(f"Connection {PROJECT_ID}.{LOCATION}.{CONNECTION_NAME} already exists — reusing it.")

# ---- 3. Extract the connection's service account dynamically (never hardcoded) ----
raw = !bq show --format=prettyjson --connection {PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}
connection_info = json.loads("\n".join(raw))
embedding_sa_email = connection_info["cloudResource"]["serviceAccountId"]
print("Connection service account:", embedding_sa_email)

# ---- 4. Grant the role at project level (idempotent -- safe to re-run) ----
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{embedding_sa_email}" \
    --role="roles/aiplatform.user" --quiet --condition=None

CONNECTION_ID = f"{PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}"

# ---- 5. Retry model creation with backoff instead of guessing a fixed sleep ----
# IAM/API-enablement propagation timing is inherently unpredictable, so retrying the actual
# operation until it succeeds is more robust than any fixed sleep duration.
max_attempts = 6
for attempt in range(1, max_attempts + 1):
    try:
        run_query(f"""
        CREATE OR REPLACE MODEL {TBL("embedding_model")}
        REMOTE WITH CONNECTION `{CONNECTION_ID}`
        OPTIONS (endpoint = 'text-embedding-004')
        """, label=f"Create embedding model (attempt {attempt})", expect_rows=False)
        print("Embedding model created successfully.")
        break
    except (BadRequest, NotFound) as e:
        wait = min(15 * attempt, 90)
        print(f"Attempt {attempt} failed ({str(e)[:150]}...) — retrying in {wait}s.")
        if attempt == max_attempts:
            raise
        time.sleep(wait)

Operation "operations/acat.p2-406774515547-aefbe3af-74d2-4399-836d-0dc039f17341" finished successfully.
Connection himanshugcpproject.US.embedding_conn already exists — reusing it.
Connection service account: bqcx-406774515547-wpxr@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Updated IAM policy for project [himanshugcpproject].
bindings:
- members:
  - serviceAccount:bqcx-406774515547-wpxr@gcp-sa-bigquery-condel.iam.gserviceaccount.com
  role: roles/aiplatform.user
- members:
  - serviceAccount:service-406774515547@gcp-sa-artifactregistry.iam.gserviceaccount.com
  role: roles/artifactregistry.serviceAgent
- members:
  - serviceAccount:406774515547@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.builds.builder
- members:
  - serviceAccount:service-406774515547@gcp-sa-cloudbuild.iam.gserviceaccount.com
  role: roles/cloudbuild.serviceAgent
- members:
  - serviceAccount:406774515547@cloudservices.gserviceaccount.com
  role: roles/compute.instanceGroupManagerServiceAgent
- member


> Check it in the console: **training_demo > product_embeddings** → **Schema** tab shows
> `embedding` as a `FLOAT64` array with a fixed dimension count — worth pointing out this is a
> completely ordinary BigQuery column, not a special vector type. **Vertex AI > Model Garden** →
> search `text-embedding-004` to see the underlying model's card and current pricing.



## Step 12 (Module 2, Part 2): Remote Functions & Connections

### Problem Statement
BigQuery's built-in SQL functions can't call an arbitrary external service — a currency-conversion
API, a company-internal validation rule implemented in Python. **Remote functions** bridge SQL to
an external HTTP endpoint (typically Cloud Functions or Cloud Run), reusing the same
**BigQuery Connection** resource type we saw in Steps 8.2 and 11.

**Why this is reference-only in this lab:** it requires a deployed HTTP endpoint outside BigQuery
entirely — genuinely out of scope for a dataset-focused lab, but the syntax and setup steps are
worth walking through since they reuse concepts (connections, IAM on the connection's service
account) already introduced above.


In [64]:
reference_sql = f"""
-- Step 1: create a Cloud resource connection (same mechanism as Steps 8.2/11)
-- bq mk --connection --connection_type=CLOUD_RESOURCE --location={LOCATION} remote_conn

-- Step 2: grant the connection's service account the Cloud Functions Invoker role
-- on your deployed endpoint (via IAM & Admin > IAM in the console)

-- Step 3: register the remote function
CREATE OR REPLACE FUNCTION {TBL("convert_to_usd")}(amount FLOAT64, currency STRING)
RETURNS FLOAT64
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.remote_conn`
OPTIONS (endpoint = 'https://YOUR_CLOUD_FUNCTION_URL')

-- Step 4: call it exactly like any built-in SQL function
-- SELECT order_id, convert_to_usd(total_amount, 'INR') AS amount_usd FROM orders
"""
print("Reference syntax only (not executed) -- deploy an endpoint first:\n")
print(reference_sql)


Reference syntax only (not executed) -- deploy an endpoint first:


-- Step 1: create a Cloud resource connection (same mechanism as Steps 8.2/11)
-- bq mk --connection --connection_type=CLOUD_RESOURCE --location=US remote_conn

-- Step 2: grant the connection's service account the Cloud Functions Invoker role
-- on your deployed endpoint (via IAM & Admin > IAM in the console)

-- Step 3: register the remote function
CREATE OR REPLACE FUNCTION `himanshugcpproject.ecommers.convert_to_usd`(amount FLOAT64, currency STRING)
RETURNS FLOAT64
REMOTE WITH CONNECTION `himanshugcpproject.US.remote_conn`
OPTIONS (endpoint = 'https://YOUR_CLOUD_FUNCTION_URL')

-- Step 4: call it exactly like any built-in SQL function
-- SELECT order_id, convert_to_usd(total_amount, 'INR') AS amount_usd FROM orders




> Check it in the console: **BigQuery > SQL workspace > External connections** (left nav
> under your project) → any connection created via `bq mk --connection` (for BigLake, vector
> search, or remote functions) shows up here in one place, with its service account email — the
> account you'd grant IAM roles to for whatever external resource it needs to reach.



## Step 13 (Module 2, Part 2): Data Lineage With Dataplex & the BigQuery Lineage API

### Problem Statement
A stakeholder asks: "where did the number in `mv_region_daily_revenue` actually come from?" Every
`CREATE TABLE ... AS SELECT`, view, and materialized view we've built in this notebook has an
implicit dependency chain — `orders` -> `v_completed_order_details` -> ... — and manually tracing
that by re-reading SQL doesn't scale past a handful of tables. **Data lineage** (part of Dataplex/
Knowledge Catalog) captures this automatically.

### 13.1 Automatic Capture Requires Only Enabling the API — No Code
Every `CREATE TABLE AS SELECT`, view, and materialized view we've already run in this notebook is
automatically eligible for lineage capture — since we enabled `datalineage.googleapis.com` in
Step 0, BigQuery has been recording these relationships already, in the background.

**Honest caveat:** lineage can take anywhere from a few minutes up to 24 hours to actually appear
— this is not instantaneous, so don't expect to see a freshly-run query's lineage show up
immediately when demoing live.

### 13.2 Query Lineage Programmatically


In [65]:
from google.cloud import datacatalog_lineage_v1

lineage_client = datacatalog_lineage_v1.LineageClient()

target_table = (
    f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}"
    f"/tables/v_completed_order_details"
)

request = datacatalog_lineage_v1.SearchLinksRequest(
    parent=f"projects/{PROJECT_ID}/locations/{LOCATION.lower()}",
    target=datacatalog_lineage_v1.EntityReference(fully_qualified_name=target_table),
)

try:
    links = list(lineage_client.search_links(request=request))
    if links:
        print(f"Found {len(links)} upstream link(s) for v_completed_order_details:")
        for link in links:
            print(f"  {link.source.fully_qualified_name}  ->  {link.target.fully_qualified_name}")
    else:
        print("No lineage links found yet — this is expected if it's been less than a few minutes")
        print("since the relevant queries ran. Lineage capture can take up to 24 hours to appear.")
except Exception as e:
    print(f"Lineage query failed (often just means nothing has propagated yet): {e}")


No lineage links found yet — this is expected if it's been less than a few minutes
since the relevant queries ran. Lineage capture can take up to 24 hours to appear.



> Check it in the console: **training_demo > v_completed_order_details** → the **Lineage**
> tab (next to Schema/Details/Preview) shows a visual graph — `customers` and `orders` feeding
> into this view, exactly matching what the API call above returns as text. This tab is genuinely
> the best way to demo this feature live, since the graph is immediately legible where the raw API
> output is not.

**On Dataplex/Knowledge Catalog's role:** lineage is a feature *of* Dataplex (now marketed as part
of Knowledge Catalog, following the same April 2026 rename covered in the Module 4 governance
notebook) — BigQuery, Dataflow, Dataproc, and Vertex AI all report into this same underlying
lineage graph, so a pipeline spanning multiple GCP services shows up as one connected graph, not
separate ones per service.



## Step 14 (Module 2, Part 2): Row-Level Security

### Problem Statement
The APAC regional team should only ever see APAC orders — even though they need query access to
the same `orders` table everyone else uses. Duplicating the table per region doesn't scale and
drifts out of sync; **Row-Level Security** filters rows per-principal on the *same* table.

Set `STUDENT_EMAIL` above to a real user/group in your domain, and `RUN_RLS_STEP = True`, before
running — row access policies grant to real IAM identities, so this isn't meaningful against a
placeholder email.


**Expected outcome (only if `RUN_RLS_STEP = True` with a real `STUDENT_EMAIL`):** the policy creation prints no rows. There's no visible change when *you* (the table owner) query `orders` afterward — RLS only filters rows for the specific principal(s) named in `GRANT TO`, not for everyone.

In [67]:
import requests
import google.auth
import google.auth.transport.requests

# Get your REAL authenticated identity reliably (works in Colab regardless of gcloud CLI state)
credentials, _ = google.auth.default()
credentials.refresh(google.auth.transport.requests.Request())
resp = requests.get(
    "https://www.googleapis.com/oauth2/v1/userinfo",
    headers={"Authorization": f"Bearer {credentials.token}"},
)
STUDENT_EMAIL = resp.json()["email"]
print("Using your own account for the RLS demo:", STUDENT_EMAIL)

if RUN_RLS_STEP:
    run_query(f"""
    CREATE OR REPLACE ROW ACCESS POLICY apac_only
    ON {TBL("orders")}
    GRANT TO ('user:{STUDENT_EMAIL}')
    FILTER USING (region = 'APAC')
    """, label="Create row access policy", expect_rows=False)
    print("Policy created. Query orders now — as YOUR account, you should only see APAC rows.")
else:
    print("RUN_RLS_STEP is False — set the toggle above to run this live.")

Using your own account for the RLS demo: himanshu.rathi@talktotechie.com
--- Create row access policy ---
Policy created. Query orders now — as YOUR account, you should only see APAC rows.



> Check it in the console: **training_demo > orders** → **Details** tab → a
> **Row access policies** section lists `apac_only` with its filter predicate and granted
> principals — visible to anyone with permission to manage the table, even without querying as
> the restricted user themselves.



## Step 15 (Module 2, Part 2): Encryption Options — AEAD & Dynamic Data Masking

### Problem Statement
Two different questions, two different mechanisms: "can we encrypt a column's *stored* value so
even someone with table access can't read it without a key" (AEAD), versus "can we show most users
a masked version of a column while letting authorized users see the real value" (dynamic data
masking, built on the policy tags from the Module 4 governance notebook).

### 15.1 Column-Level Encryption (AEAD)
Encrypt the `email` column at query time using a session keyset. Cheap to run, no extra GCP setup.


**Expected outcome:** one result table prints with `customer_id` and an `encrypted_email` column containing unreadable binary/base64-looking bytes — confirming the encryption ran, even though this specific keyset is discarded the moment the query ends (see the caveat below).

In [68]:
run_query(f"""
DECLARE keyset BYTES DEFAULT KEYS.NEW_KEYSET('AEAD_AES_GCM_256');

SELECT customer_id,
       AEAD.ENCRYPT(keyset, email, CAST(customer_id AS STRING)) AS encrypted_email
FROM {TBL("customers")}
LIMIT 10
""", label="AEAD column encryption")


--- AEAD column encryption ---


,customer_id,encrypted_email
0,3,b'\x01h\\\xd5\x87d<\x17\x12\xb89\\B\x17g9\n\xd...
1,18,b'\x01h\\\xd5\x87u\xb0-\xae\xc6\x06\x8d\x87\xc...
2,28,"b""\x01h\\\xd5\x87\x00@\x86\xb4\x1e|`\xb9\xacy\..."
3,8,b'\x01h\\\xd5\x87\x85ElcA_\xda\xf5\xacw\x88|Pi...
4,13,b'\x01h\\\xd5\x87\xfej\xf3\x94\x9a\x06W\xcb\x8...
5,23,b'\x01h\\\xd5\x87\xe6\x89\xfa\xb5\xb7\xf8\xe7\...
6,17,b'\x01h\\\xd5\x87Z\xf3.\x87\xcc\r\xee\xe0\xbf\...
7,22,b'\x01h\\\xd5\x87\n}\xddE\xf4]\xadK\xff?T\xddt...
8,7,b'\x01h\\\xd5\x87TV\xa1\x9b\xda\x81\xd8\xcd\xd...
9,12,b'\x01h\\\xd5\x87eBx\xf1\xf6\x0c\xe3\xe0\xd8\x...


QueryJob<project=himanshugcpproject, location=US, id=17dab145-2c08-4418-942a-352eefd8fc26>


**Teaching point:** the keyset here is generated inline and discarded when the query ends — in
production you'd store the keyset in Cloud KMS and reference it consistently so encrypted values
can actually be decrypted later. This demo shows the encryption mechanics, not full key-management
practice.

### 15.2 Dynamic Data Masking — the Alternative to AEAD
**How this differs from AEAD:** AEAD produces genuinely unreadable ciphertext requiring an
explicit key to reverse. **Dynamic data masking** instead shows a *masked but still useful* value
(e.g. `XXX-XXX-1234` for a phone number) to unauthorized users automatically, based on the same
**policy tags** covered in the Module 4 Security & Governance notebook — no query rewriting
needed, since masking is applied transparently based on who's asking.


In [69]:
reference_masking_sql = f"""
-- Reference only — requires a taxonomy and policy tag already created (see the Module 4
-- governance notebook, Section 3), then applying a masking rule to a tagged column:

ALTER TABLE {TBL("customers")}
ALTER COLUMN email
SET OPTIONS (
  policy_tags = ['projects/{PROJECT_ID}/locations/{LOCATION}/taxonomies/TAXONOMY_ID/policyTags/POLICY_TAG_ID']
);

-- Then, in the taxonomy's masking rule configuration (set via the console or Data Policy API),
-- specify a masking type such as 'EMAIL_MASK' or 'DEFAULT_MASKING_VALUE' -- unauthorized queriers
-- then see 'XXXXX@XXXXX.com' automatically, without the query itself changing at all.
"""
print(reference_masking_sql)



-- Reference only — requires a taxonomy and policy tag already created (see the Module 4
-- governance notebook, Section 3), then applying a masking rule to a tagged column:

ALTER TABLE `himanshugcpproject.ecommers.customers`
ALTER COLUMN email
SET OPTIONS (
  policy_tags = ['projects/himanshugcpproject/locations/US/taxonomies/TAXONOMY_ID/policyTags/POLICY_TAG_ID']
);

-- Then, in the taxonomy's masking rule configuration (set via the console or Data Policy API),
-- specify a masking type such as 'EMAIL_MASK' or 'DEFAULT_MASKING_VALUE' -- unauthorized queriers
-- then see 'XXXXX@XXXXX.com' automatically, without the query itself changing at all.




> Check it in the console: **Governance > Knowledge Catalog > Policy tags** → any taxonomy
> → a policy tag's **Data masking** tab lets you pick a masking rule (Default masking value, Email
> mask, Date year mask, etc.) directly — this is genuinely a console-first feature since defining
> a taxonomy visually is much easier than via API for a one-off classification.

**When to use which:** AEAD when you need true reversible encryption for compliance (e.g. GDPR
"right to be forgotten" via key deletion); dynamic masking when most users should simply never see
raw sensitive values by default, with a smaller trusted group granted the unmasking role.



## Cleanup

This notebook created a real cost surface across three modules — the dataset (with every table,
view, materialized view, and model inside it), the GCS bucket (including any Iceberg warehouse
files), any BigQuery connections, and — only if you actually ran Step 8.3.3 for real — a slot
reservation and capacity commitment. Guarded by `CONFIRM_DELETE`.


In [74]:
CONFIRM_DELETE = True  #@param {type:"boolean"}

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing will be deleted. "
          "Set CONFIRM_DELETE = True above and re-run this cell when you're ready.")


In [75]:
if CONFIRM_DELETE:
    # Row access policies and IAM bindings live on the tables themselves, so deleting the
    # dataset below removes them too -- no separate cleanup step needed for those.
    try:
        bq_client.delete_dataset(
            f"{PROJECT_ID}.{DATASET_ID}", delete_contents=True, not_found_ok=True
        )
        print(f"Deleted dataset {DATASET_ID} (every table, view, materialized view, and model inside it).")
    except Exception as e:
        print("Dataset cleanup skipped/failed:", e)


Deleted dataset ecommers (every table, view, materialized view, and model inside it).


In [76]:
if CONFIRM_DELETE:
    try:
        bucket = storage_client.bucket(BUCKET_NAME)
        bucket.delete(force=True)
        print(f"Deleted bucket {BUCKET_NAME} (including any Iceberg warehouse files).")
    except Exception as e:
        print("Bucket cleanup skipped/failed:", e)


Deleted bucket mybigqueryadvancebucket007 (including any Iceberg warehouse files).


In [77]:
if CONFIRM_DELETE:
    # Clean up any BigQuery connections created for BigLake/Iceberg, vector search, or remote functions
    connection_names_to_check = ["iceberg_conn", "embedding_conn", "remote_conn"]
    for conn_name in connection_names_to_check:
        try:
            !bq rm --connection --project_id={PROJECT_ID} --location={LOCATION} -f {conn_name}
            print(f"Deleted connection: {conn_name}")
        except Exception as e:
            print(f"Connection {conn_name} cleanup skipped (may not have been created): {e}")


Deleted connection: iceberg_conn
Deleted connection: embedding_conn
BigQuery error in rm operation: Not found: Connection
projects/himanshugcpproject/locations/US/connections/remote_conn
Deleted connection: remote_conn


In [78]:
if CONFIRM_DELETE:
    if RUN_RESERVATION_STEP:
        print("RUN_RESERVATION_STEP was True at some point in this session -- if you actually")
        print("executed the reservation/commitment commands from Step 8.3.3 for real (they were")
        print("commented out by default), remove assignment -> reservation -> commitment, in that")
        print("order, via:")
        print(f"  bq rm --project_id={PROJECT_ID} --location={LOCATION} --reservation_assignment=true ASSIGNMENT_ID")
        print(f"  bq rm --project_id={PROJECT_ID} --location={LOCATION} --reservation=true training-demo-reservation")
        print(f"  bq rm --project_id={PROJECT_ID} --location={LOCATION} --capacity_commitment=true COMMITMENT_ID")
        print("Capacity commitments cannot be deleted before their commitment period ends (annual/3-year).")
    else:
        print("No reservation/commitment was created in this session — nothing to clean up there.")

    print("\nCleanup complete.")


RUN_RESERVATION_STEP was True at some point in this session -- if you actually
executed the reservation/commitment commands from Step 8.3.3 for real (they were
commented out by default), remove assignment -> reservation -> commitment, in that
order, via:
  bq rm --project_id=himanshugcpproject --location=US --reservation_assignment=true ASSIGNMENT_ID
  bq rm --project_id=himanshugcpproject --location=US --reservation=true training-demo-reservation
  bq rm --project_id=himanshugcpproject --location=US --capacity_commitment=true COMMITMENT_ID
Capacity commitments cannot be deleted before their commitment period ends (annual/3-year).

Cleanup complete.



> Final console check: **BigQuery > SQL workspace > Explorer** should no longer list the
> `training_demo` dataset at all. **Cloud Storage > Buckets** should no longer list your bucket.
> **BigQuery > SQL workspace > External connections** should show none of `iceberg_conn`,
> `embedding_conn`, or `remote_conn` remaining. **BigQuery > Administration > Workload management**
> should reflect no reservations/commitments unless you deliberately created and are intentionally
> keeping one.
